# 00 — Environment Setup Test

Sanity check: can we reach `nba_api`, find Luka Dončić in the static player registry, and fetch his current season stats?

If the final cell prints a non-empty DataFrame, the environment is wired up correctly and Phase 1 can begin.

In [3]:
#%pip install nba_api

In [4]:
# Step 1 — verify nba_api is importable
import nba_api
print(f"nba_api version: {nba_api.__version__}")

nba_api version: 1.11.4


In [5]:
# Step 2 — find Luka Dončić's player ID from static data (no network call)
from nba_api.stats.static import players

results = players.find_players_by_full_name("Luka Doncic")
if not results:
    # Accent variant
    results = players.find_players_by_full_name("Luka Don")

luka = results[0]
LUKA_ID = luka["id"]
print(f"Found: {luka['full_name']}  |  ID: {LUKA_ID}  |  Active: {luka['is_active']}")

Found: Luka Dončić  |  ID: 1629029  |  Active: True


In [6]:
# Step 3 — fetch Luka's career stats (hits stats.nba.com — requires network)
import time
import pandas as pd
from nba_api.stats.endpoints import playercareerstats

time.sleep(0.6)  # respect rate limit before first call

career = playercareerstats.PlayerCareerStats(
    player_id=LUKA_ID,
    per_mode36="PerGame",
    timeout=30,
)

df = career.get_data_frames()[0]
print(f"Rows returned: {len(df)}")
df.head()

Rows returned: 10


,PLAYER_ID,SEASON_ID,LEAGUE_ID,TEAM_ID,TEAM_ABBREVIATION,PLAYER_AGE,GP,GS,MIN,FGM,...,FT_PCT,OREB,DREB,REB,AST,STL,BLK,TOV,PF,PTS
0,1629029,2018-19,00,1610612742,DAL,20.0,72,72,32.2,7.0,...,0.713,1.2,6.6,7.8,6.0,1.1,0.3,3.4,1.9,21.2
1,1629029,2019-20,00,1610612742,DAL,21.0,61,61,33.6,9.5,...,0.758,1.3,8.1,9.4,8.8,1.0,0.2,4.3,2.5,28.8
2,1629029,2020-21,00,1610612742,DAL,22.0,66,66,34.3,9.8,...,0.730,0.8,7.2,8.0,8.6,1.0,0.5,4.3,2.3,27.7
3,1629029,2021-22,00,1610612742,DAL,23.0,65,65,35.4,9.9,...,0.744,0.9,8.3,9.1,8.7,1.2,0.6,4.5,2.2,28.4
4,1629029,2022-23,00,1610612742,DAL,24.0,66,66,36.2,10.9,...,0.742,0.8,7.8,8.6,8.0,1.4,0.5,3.6,2.5,32.4


In [7]:
# Step 4 — confirm current (most recent) season row
current_row = df.sort_values("SEASON_ID").iloc[[-1]]
cols = ["SEASON_ID", "TEAM_ABBREVIATION", "GP", "PTS", "AST", "REB", "FG3_PCT"]
print("Most recent season stats (per game):")
current_row[cols]

Most recent season stats (per game):


,SEASON_ID,TEAM_ABBREVIATION,GP,PTS,AST,REB,FG3_PCT
9,2025-26,LAL,64,33.5,8.3,7.7,0.366


---
**Expected output:** A single-row DataFrame showing Luka's 2024-25 (or 2025-26) per-game averages.

If you see data, the pipeline is ready. Proceed to `01_data_exploration.ipynb`.